# EDA formalisée — Dataset Wine

## Objectif

L'objectif de cette EDA est d'explorer le dataset **Wine** disponible directement dans `scikit-learn`.

Le dataset contient **178 vins**, décrits par **13 variables numériques** liées à leurs caractéristiques chimiques, ainsi qu'une variable cible indiquant la classe du vin.

> Méthode : avant de faire l'analyse, je formule 10 questions. Chaque question sera traitée avec du code, au moins un graphique et une conclusion courte.


## Les 10 questions avant l'analyse

1. Le dataset est-il propre et complet ?
2. Les trois classes de vins sont-elles équilibrées ?
3. Comment se répartit le taux d'alcool ?
4. Quelles variables présentent le plus de dispersion ?
5. Le taux d'alcool est-il lié à la quantité de proline ?
6. Le taux d'alcool varie-t-il selon la classe de vin ?
7. Les flavanoïdes permettent-ils de différencier les classes ?
8. Quelles variables sont les plus corrélées entre elles ?
9. Les variables `alcohol` et `proline` contiennent-elles des valeurs atypiques ?
10. Les variables chimiques permettent-elles de séparer visuellement les trois classes ?


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

sns.set_theme(style="whitegrid")

wine = load_wine(as_frame=True)
df = wine.frame.copy()

class_names = {i: name for i, name in enumerate(wine.target_names)}
df["target_name"] = df["target"].map(class_names)

df.head()


## Question 1 — Le dataset est-il propre et complet ?

Je vérifie les dimensions, les types, les valeurs manquantes et les doublons.


In [ ]:
print("Dimensions :", df.shape)
print("\nTypes :")
print(df.dtypes)

print("\nValeurs manquantes :")
print(df.isna().sum())

print("\nNombre de doublons :", df.duplicated().sum())

missing = df.isna().sum()
missing.plot(kind="bar", figsize=(10, 4), title="Nombre de valeurs manquantes par variable")
plt.ylabel("Nombre de valeurs manquantes")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### Conclusion

Le dataset contient **178 lignes et 15 colonnes**, dont 13 variables explicatives, la cible numérique et son libellé. Il n'y a **aucune valeur manquante ni aucun doublon**, donc les données sont suffisamment propres pour poursuivre l'analyse sans étape d'imputation ou de déduplication.

## Question 2 — Les trois classes de vins sont-elles équilibrées ?

Je regarde le nombre d'observations dans chaque classe afin de savoir si une classe est surreprésentée.


In [ ]:
class_counts = df["target_name"].value_counts().sort_index()
print(class_counts)

class_counts.plot(kind="bar", figsize=(7, 4), title="Nombre de vins par classe")
plt.xlabel("Classe")
plt.ylabel("Nombre de vins")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### Conclusion

Les classes contiennent respectivement **59, 71 et 48 observations**. Elles ne sont donc pas parfaitement équilibrées, mais l'écart reste raisonnable : aucune classe ne domine suffisamment pour rendre l'exploration impossible.

## Question 3 — Comment se répartit le taux d'alcool ?

Je cherche à voir la forme de la distribution et les valeurs les plus fréquentes.


In [ ]:
print(df["alcohol"].describe())

plt.figure(figsize=(8, 4))
sns.histplot(df["alcohol"], bins=15, kde=True)
plt.title("Distribution du taux d'alcool")
plt.xlabel("Taux d'alcool")
plt.ylabel("Nombre de vins")
plt.tight_layout()
plt.show()


### Conclusion

Le taux d'alcool a une médiane d'environ **13,05** et la moitié des valeurs se situe entre **12,36 et 13,68**. La distribution couvre donc une plage assez resserrée, avec quelques valeurs plus faibles ou plus élevées.

## Question 4 — Quelles variables présentent le plus de dispersion ?

Une variable très dispersée peut révéler des différences importantes entre les observations. Je compare ici les écarts-types des variables numériques.


In [ ]:
numeric_cols = df.select_dtypes(include="number").drop(columns=["target"])
std_values = numeric_cols.std().sort_values(ascending=False)

print(std_values)

plt.figure(figsize=(9, 5))
std_values.plot(kind="bar")
plt.title("Dispersion des variables numériques")
plt.xlabel("Variable")
plt.ylabel("Écart-type")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()


### Conclusion

La **proline** est de très loin la variable qui présente la plus forte dispersion, avec un écart-type d'environ **315**. Il faut toutefois faire attention à l'échelle des variables : un grand écart-type ne signifie pas automatiquement qu'une variable est plus importante.

## Question 5 — Le taux d'alcool est-il lié à la quantité de proline ?

Je représente les deux variables sur un nuage de points et je calcule leur corrélation de Pearson.


In [ ]:
correlation = df["alcohol"].corr(df["proline"])
print(f"Corrélation alcohol / proline : {correlation:.3f}")

plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="alcohol", y="proline", hue="target_name")
plt.title("Relation entre le taux d'alcool et la proline")
plt.xlabel("Taux d'alcool")
plt.ylabel("Proline")
plt.legend(title="Classe")
plt.tight_layout()
plt.show()


### Conclusion

La corrélation entre `alcohol` et `proline` est d'environ **0,64**, ce qui correspond à une relation positive assez marquée. Les vins ayant davantage d'alcool ont donc tendance à avoir davantage de proline, même si la relation n'est pas parfaite.

## Question 6 — Le taux d'alcool varie-t-il selon la classe de vin ?

Je compare les distributions du taux d'alcool pour les trois classes.


In [ ]:
alcohol_by_class = df.groupby("target_name")["alcohol"].mean().sort_values(ascending=False)
print(alcohol_by_class)

plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="target_name", y="alcohol")
sns.stripplot(data=df, x="target_name", y="alcohol", alpha=0.35, color="black")
plt.title("Taux d'alcool selon la classe")
plt.xlabel("Classe")
plt.ylabel("Taux d'alcool")
plt.tight_layout()
plt.show()


### Conclusion

La moyenne du taux d'alcool est d'environ **13,74 pour la classe 0**, **12,28 pour la classe 1** et **13,15 pour la classe 2**. La classe 1 se distingue donc par un taux d'alcool globalement plus faible que les deux autres classes.

## Question 7 — Les flavanoïdes permettent-ils de différencier les classes ?

Les flavanoïdes sont une variable intéressante à comparer entre les groupes.


In [ ]:
flav_mean = df.groupby("target_name")["flavanoids"].mean().sort_values(ascending=False)
print(flav_mean)

plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="target_name", y="flavanoids")
plt.title("Flavanoïdes selon la classe")
plt.xlabel("Classe")
plt.ylabel("Flavanoïdes")
plt.tight_layout()
plt.show()


### Conclusion

Les différences sont très nettes : la moyenne des flavanoïdes est d'environ **2,98 pour la classe 0**, **2,08 pour la classe 1** et seulement **0,78 pour la classe 2**. Cette variable semble donc particulièrement informative pour distinguer les classes.

## Question 8 — Quelles variables sont les plus corrélées entre elles ?

Je calcule la matrice de corrélation afin d'identifier les relations fortes et les variables potentiellement redondantes.


In [ ]:
corr_matrix = numeric_cols.corr()

plt.figure(figsize=(12, 9))
sns.heatmap(corr_matrix, cmap="coolwarm", center=0)
plt.title("Matrice de corrélation")
plt.tight_layout()
plt.show()

corr_pairs = (
    corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    .stack()
    .sort_values(key=lambda s: s.abs(), ascending=False)
)

print("10 corrélations les plus fortes :")
print(corr_pairs.head(10))


### Conclusion

La matrice montre plusieurs relations importantes entre variables. La relation entre **flavanoïdes et OD280/OD315** est particulièrement forte, ce qui indique que ces variables portent en partie une information similaire ; il faudra en tenir compte dans une éventuelle modélisation.

## Question 9 — Les variables `alcohol` et `proline` contiennent-elles des valeurs atypiques ?

Je cherche les valeurs situées au-delà des bornes classiques de l'IQR (règle des 1,5 × IQR).


In [ ]:
def outlier_info(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = series[(series < lower) | (series > upper)]
    return q1, q3, lower, upper, len(outliers)

for col in ["alcohol", "proline"]:
    q1, q3, lower, upper, n = outlier_info(df[col])
    print(f"{col}: Q1={q1:.2f}, Q3={q3:.2f}, bornes=[{lower:.2f}, {upper:.2f}], outliers={n}")

plt.figure(figsize=(9, 4))
sns.boxplot(data=df[["alcohol", "proline"]])
plt.title("Détection visuelle des valeurs atypiques")
plt.tight_layout()
plt.show()


### Conclusion

La règle de l'IQR permet d'identifier quelques observations atypiques, surtout sur la **proline**, qui possède une dispersion beaucoup plus importante. Ces valeurs ne doivent pas être supprimées automatiquement : elles peuvent correspondre à de vrais vins et doivent être étudiées avant toute transformation.

## Question 10 — Les variables chimiques permettent-elles de séparer visuellement les trois classes ?

Je standardise les variables puis j'utilise une ACP (PCA) pour projeter les 13 dimensions dans un espace à deux dimensions.


In [ ]:
X = df[numeric_cols.columns]
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "target_name": df["target_name"]
})

print("Variance expliquée par PC1 :", f"{pca.explained_variance_ratio_[0]:.3f}")
print("Variance expliquée par PC2 :", f"{pca.explained_variance_ratio_[1]:.3f}")
print("Variance expliquée cumulée :", f"{pca.explained_variance_ratio_.sum():.3f}")

plt.figure(figsize=(9, 6))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="target_name", s=70)
plt.title("Projection PCA des vins")
plt.xlabel("Composante principale 1")
plt.ylabel("Composante principale 2")
plt.tight_layout()
plt.show()


### Conclusion

Les deux premières composantes principales expliquent environ **55,4 % de la variance totale**. La projection montre une séparation visuelle assez nette des trois classes, ce qui confirme que les caractéristiques chimiques contiennent une information forte sur la classe du vin.

# Bilan de l'EDA

Cette analyse montre que le dataset est propre et exploitable, avec des classes relativement équilibrées. Plusieurs variables semblent  liées à la classe, comme les **flavanoïdes**, la **proline** et le **taux d'alcool**.

La PCA confirme qu'une partie importante de l'information peut être représentée en deux dimensions et que les classes sont relativement bien séparables. Pour aller plus loin, on pourrait tester un modèle de classification et comparer son importance des variables avec les observations faites pendant cette EDA.
